# 🍎 HyperSnake-AI: 1-Click Vectorized RL Engine 🐍

Welcome to the autonomous training environment for **HyperSnake-AI**.
This notebook deploys a state-of-the-art Proximal Policy Optimization (PPO) agent to solve a 9x10 Snake grid using a highly optimized, fully GPU-resident architecture.

### ⚡ Performance Highlights:
* **Vectorized VRAM Engine:** 4,096 parallel environments running entirely on the GPU.
* **Spatial Awareness:** Coordinate Convolutions (CoordConv) and Dilated Residual Blocks.
* **Compute Efficiency:** Native FP16 Mixed Precision & PyTorch 2.0 Compiled GAE.

### 📖 How to Train Your Agent:
1. **Allocate Hardware:** Go to `Runtime` > `Change runtime type` > Select **T4 GPU**.
2. **Ignite:** Press `Ctrl + F9` (or `Cmd + F9` on Mac) to run all cells sequentially.
3. **Monitor Live:** Scroll down to the TensorBoard cell to watch the agent learn in real-time. Checkpoints are automatically secured to your Google Drive.

---
🔗 **GitHub:** [siddhartha399/HyperSnake-AI](https://github.com/siddhartha399/HyperSnake-AI)

In [ ]:
# @title Step 1: Engine Initialization & Hardware Allocation
import os
import torch
import subprocess

def setup_environment():
    print("="*60)
    print("🚀 INITIALIZING HYPERSNAKE-AI ENGINE")
    print("="*60)

    if not torch.cuda.is_available():
        raise SystemError("❌ GPU not detected. Please go to Runtime > Change runtime type and select T4 GPU.")

    device_name = torch.cuda.get_device_name(0)
    print(f"[OK] Hardware Allocated: {device_name}")
    print(f"[OK] PyTorch Version: {torch.__version__} (Compile-ready)")

    # Target Repository
    REPO_URL = "https://github.com/siddhartha399/HyperSnake-AI.git"
    REPO_NAME = "HyperSnake-AI"

    if not os.path.exists(REPO_NAME):
        print(f"\n[INFO] Cloning Repository from {REPO_URL}...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"\n[OK] Repository '{REPO_NAME}' already exists.")

    os.chdir(REPO_NAME)
    print(f"[OK] System routed to: {os.getcwd()}")

    if os.path.exists("requirements.txt"):
        print("[INFO] Locking in dependencies...")
        subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"], check=True)
        print("[OK] Environment secured.")

setup_environment()

In [ ]:
# @title Step 2: Mount Google Drive (Persistent Checkpoints)
from google.colab import drive
import os

print("="*60)
print("🔒 SECURING CHECKPOINT STORAGE")
print("="*60)

# Connect to Google Drive to ensure models aren't lost if Colab disconnects
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/Snake_AI_PPO/Checkpoints_9x10"
LOG_DIR = "/content/drive/MyDrive/Snake_AI_PPO/logs_9x10"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"[OK] Neural Weights routed to: {CHECKPOINT_DIR}")
print(f"[OK] Telemetry routed to: {LOG_DIR}")

In [ ]:
# @title Step 3: Launch Live Telemetry (TensorBoard)
# Load the TensorBoard extension
%load_ext tensorboard

# Launch the dashboard pointing to the live log directory
%tensorboard --logdir /content/drive/MyDrive/Snake_AI_PPO/logs_9x10

In [ ]:
# @title Step 4: Ignite Vectorized Training Loop
print("="*60)
print("🔥 INITIATING PPO LOOP")
print("="*60)
print("Note: The training loop is infinite. Press the 'Stop' button (◾) on this cell to safely halt training. The engine will trigger an emergency checkpoint save before shutting down.")
print("="*60 + "\n")

# Execute the hyper-optimized training script
!python train.py

<div align="left">

# HyperSnake-AI Visualizer

### Direct Inference & Real-Time Visualization directly in Colab

Trainings are complete, and now it is time to evaluate the policy. This cell contains the **entire architecture, environment, and inference engine** condensed into a single, highly optimized script for seamless execution within a Jupyter notebook.

</div>

---

## 🚀 How to Run

This script is entirely self-contained. It mounts your Google Drive, loads the trained CoordConv weights, and renders the environment frame-by-frame directly in the output cell using ANSI color coding.

**Configuration Required:**
The only variable you need to modify is the path to your trained weights. Ensure your checkpoint is stored in your Google Drive and update the path in the `Config` class:

```python
BEST_MODEL_PATH = "/content/drive/MyDrive/path/to/your/ppo_snake_9x10_best.pt"

In [ ]:
"""
===============================================================================
VISUALIZER
===============================================================================

A self-contained, GPU-accelerated inference and rendering pipeline for evaluating
the fully trained HyperSnake-AI agent directly in Colab.

===============================================================================
"""

import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.distributions.categorical import Categorical
from IPython.display import clear_output
from google.colab import drive

# --- 1. SYSTEM INITIALIZATION ---
drive.mount('/content/drive')

class Config:
    GRID_H          = 9
    GRID_W          = 10
    NUM_ENVS        = 1      # Isolated environment for focused visualization
    DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # [!] TARGET CHECKPOINT: Update this path to your highest-scoring model
    BEST_MODEL_PATH = "/content/drive/MyDrive/Snake_AI_PPO/Checkpoints_9x10/ppo_snake_9x10_best.pt"

# --- 2. CORE ARCHITECTURE ---
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class ResidualBlock(nn.Module):
    def __init__(self, channels: int, dilation: int = 1):
        super().__init__()
        padding = dilation
        self.conv1 = layer_init(nn.Conv2d(channels, channels, 3, padding=padding, dilation=dilation))
        self.conv2 = layer_init(nn.Conv2d(channels, channels, 3, padding=padding, dilation=dilation))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.conv2(F.relu(self.conv1(x))) + x)

class PPOAgent(nn.Module):
    def __init__(self, h=Config.GRID_H, w=Config.GRID_W):
        super().__init__()
        # Observation Channels: Head, Body, Tail, Food, Danger + Coord X + Coord Y
        self.extractor = nn.Sequential(
            layer_init(nn.Conv2d(7, 64, 3, padding=1)),
            nn.ReLU(),
            ResidualBlock(64, dilation=1),
            ResidualBlock(64, dilation=1),
            ResidualBlock(64, dilation=1),
            ResidualBlock(64, dilation=1),
            ResidualBlock(64, dilation=2),
            ResidualBlock(64, dilation=2)
        )

        # Actor Head
        self.actor_conv = layer_init(nn.Conv2d(64, 2, 1))
        self.actor_linear = layer_init(nn.Linear(2 * h * w, 4), std=0.01)

        # Critic Head
        self.critic_conv = layer_init(nn.Conv2d(64, 1, 1))
        self.critic_hidden = layer_init(nn.Linear(1 * h * w, 256), std=1.0)
        self.critic_out = layer_init(nn.Linear(256, 1), std=1.0)

    def forward_features(self, x):
        features = self.extractor(x)

        a = F.relu(self.actor_conv(features))
        a = torch.flatten(a, start_dim=1)
        logits = self.actor_linear(a)

        c = F.relu(self.critic_conv(features))
        c = torch.flatten(c, start_dim=1)
        v_hidden = F.relu(self.critic_hidden(c))
        value = self.critic_out(v_hidden)

        return logits, value

    def get_action_and_value(self, x, action_mask, action=None):
        logits, value = self.forward_features(x)
        huge_neg = torch.tensor(-1e8, device=Config.DEVICE)
        masked_logits = torch.where(action_mask.bool(), logits, huge_neg)
        probs = Categorical(logits=masked_logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), value

def clean_state_dict(state_dict):
    """Strips compilation prefixes from state dictionary keys."""
    return {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

# --- 3. VECTORIZED ENVIRONMENT ---
class VectorizedSnakeEnv:
    def __init__(self, num_envs, grid_h, grid_w, device):
        self.B = num_envs
        self.H = grid_h
        self.W = grid_w
        self.device = device
        self.dirs = torch.tensor([[-1, 0], [0, 1], [1, 0], [0, -1]], device=device)

        self.grid = torch.zeros((self.B, self.H, self.W), dtype=torch.int32, device=device)
        self.head = torch.zeros((self.B, 2), dtype=torch.long, device=device)
        self.food = torch.zeros((self.B, 2), dtype=torch.long, device=device)
        self.length = torch.full((self.B,), 3, dtype=torch.int32, device=device)
        self.score = torch.zeros(self.B, dtype=torch.float32, device=device)
        self.steps = torch.zeros(self.B, dtype=torch.int32, device=device)

        # Static CoordConv generation
        y_coords = torch.linspace(-1.0, 1.0, self.H, device=device).view(1, 1, self.H, 1).expand(self.B, 1, self.H, self.W)
        x_coords = torch.linspace(-1.0, 1.0, self.W, device=device).view(1, 1, 1, self.W).expand(self.B, 1, self.H, self.W)
        self.static_coords = torch.cat([y_coords, x_coords], dim=1)

        self.reset_all()

    def reset_all(self):
        self.grid.zero_()
        cy, cx = self.H // 2, self.W // 2
        self.head[:, 0] = cy
        self.head[:, 1] = cx
        self.length.fill_(3)
        self.score.zero_()
        self.steps.zero_()
        self.grid[:, cy, cx] = 3
        self.grid[:, cy, cx+1] = 2
        self.grid[:, cy, cx+2] = 1
        self._place_food(torch.ones(self.B, dtype=torch.bool, device=self.device))

    def _place_food(self, mask):
        idx = mask.nonzero(as_tuple=True)[0]
        if len(idx) == 0:
            return

        empty_mask = (self.grid[idx] == 0)
        has_empty = empty_mask.view(len(idx), -1).any(dim=1)
        valid_idx = idx[has_empty]

        if len(valid_idx) == 0:
            return

        empty_mask_valid = (self.grid[valid_idx] == 0)
        noise = torch.rand(empty_mask_valid.shape, device=self.device)
        valid_noise = torch.where(empty_mask_valid, noise, torch.tensor(-1.0, device=self.device))

        flat_noise = valid_noise.view(len(valid_idx), -1)
        best_idx = flat_noise.argmax(dim=1)

        self.food[valid_idx, 0] = best_idx // self.W
        self.food[valid_idx, 1] = best_idx % self.W

    def step(self, actions):
        self.steps += 1
        moves = self.dirs[actions]
        new_head = self.head + moves

        out_of_bounds = (new_head[:, 0] < 0) | (new_head[:, 0] >= self.H) | \
                        (new_head[:, 1] < 0) | (new_head[:, 1] >= self.W)

        safe_new_head = new_head.clone()
        safe_new_head[out_of_bounds] = 0

        b_idx = torch.arange(self.B, device=self.device)
        hit_body = (self.grid[b_idx, safe_new_head[:, 0], safe_new_head[:, 1]] > 1) & ~out_of_bounds

        died = out_of_bounds | hit_body | (self.steps >= (150 + 100 * self.score))
        survived = ~died
        ate_food = survived & (new_head[:, 0] == self.food[:, 0]) & (new_head[:, 1] == self.food[:, 1])

        self.score[ate_food] += 1
        self.length[ate_food] += 1

        won = self.length == (self.H * self.W)
        done = died | won

        survived_no_food = survived & ~ate_food
        if survived_no_food.any():
            s_idx = survived_no_food.nonzero(as_tuple=True)[0]
            mask_gt_0 = self.grid[s_idx] > 0
            self.grid[s_idx] -= mask_gt_0.to(torch.int32)

        if survived.any():
            s_idx = survived.nonzero(as_tuple=True)[0]
            self.head[s_idx] = new_head[s_idx]
            self.grid[s_idx, self.head[s_idx, 0], self.head[s_idx, 1]] = self.length[s_idx]

        needs_food = ate_food & ~won
        if needs_food.any():
            self._place_food(needs_food)

        info_scores = self.score[done].clone()
        return self.get_states(), done, info_scores, self.get_legal_masks()

    def get_states(self):
        states = torch.zeros((self.B, 5, self.H, self.W), dtype=torch.float32, device=self.device)
        b_idx = torch.arange(self.B, device=self.device)

        states[b_idx, 0, self.head[:, 0], self.head[:, 1]] = 1.0

        body_mask = (self.grid > 1)
        body_mask[b_idx, self.head[:, 0], self.head[:, 1]] = False
        states[:, 1] = body_mask.to(torch.float32)

        states[:, 2] = (self.grid == 1).to(torch.float32)
        states[b_idx, 3, self.food[:, 0], self.food[:, 1]] = 1.0

        whole_body = (self.grid > 0)
        whole_body[b_idx, self.head[:, 0], self.head[:, 1]] = False
        states[:, 4] = whole_body.to(torch.float32)

        full_states = torch.cat([states, self.static_coords], dim=1)
        return full_states.to(memory_format=torch.channels_last)

    def get_legal_masks(self):
        masks = torch.zeros((self.B, 4), dtype=torch.float32, device=self.device)
        b_idx = torch.arange(self.B, device=self.device).unsqueeze(1)
        targets = self.head.unsqueeze(1) + self.dirs.unsqueeze(0)

        valid_bounds = (targets[:, :, 0] >= 0) & (targets[:, :, 0] < self.H) & \
                       (targets[:, :, 1] >= 0) & (targets[:, :, 1] < self.W)

        safe_targets = targets.clone()
        safe_targets[~valid_bounds] = 0

        grid_vals = self.grid[b_idx, safe_targets[:, :, 0], safe_targets[:, :, 1]]
        is_legal = valid_bounds & (grid_vals <= 1)

        masks[is_legal] = 1.0
        no_moves = masks.sum(dim=1) == 0
        masks[no_moves] = 1.0

        return masks

# --- 4. ANSI RENDERER & INFERENCE ENGINE ---
def visualize_best_model():
    GREEN_BG = "\033[42m   \033[0m"
    DARK_GREEN_BG = "\033[48;5;22m   \033[0m"
    RED_BG = "\033[41m   \033[0m"
    EMPTY = " . "

    device = Config.DEVICE
    agent = PPOAgent().to(device).to(memory_format=torch.channels_last)

    print("[SYSTEM] Initializing Inference Pipeline...")
    if os.path.exists(Config.BEST_MODEL_PATH):
        checkpoint = torch.load(Config.BEST_MODEL_PATH, map_location=device, weights_only=False)
        agent.load_state_dict(clean_state_dict(checkpoint['model_state_dict']))
        agent.eval()
        print(f"[+] Spatial Checkpoint loaded securely. Best Mean Score: {checkpoint.get('best_mean_score', 'N/A'):.2f}")
    else:
        print(f"[-] Checkpoint missing at defined path:\n    {Config.BEST_MODEL_PATH}\n    Please update Config.BEST_MODEL_PATH and re-run.")
        return

    time.sleep(1)

    env = VectorizedSnakeEnv(Config.NUM_ENVS, Config.GRID_H, Config.GRID_W, device)
    obs = env.get_states()
    mask = env.get_legal_masks()

    step = 0
    while True:
        with torch.no_grad():
            action, _, _, _ = agent.get_action_and_value(obs, mask)

        obs, done, info, mask = env.step(action)
        step += 1

        head_channel = obs[0, 0].cpu().numpy()
        body_channel = obs[0, 4].cpu().numpy()
        food_channel = obs[0, 3].cpu().numpy()

        clear_output(wait=True)
        score = int(env.score[0].item())

        print(f"==========================================")
        print(f" HYPERSNAKE-AI | SCORE: {score:03d} | STEP: {step:04d}")
        print(f"==========================================")

        for r in range(Config.GRID_H):
            row_str = ""
            for c in range(Config.GRID_W):
                if head_channel[r, c] > 0:
                    row_str += GREEN_BG
                elif body_channel[r, c] > 0:
                    row_str += DARK_GREEN_BG
                elif food_channel[r, c] > 0:
                    row_str += RED_BG
                else:
                    row_str += EMPTY
            print(row_str)

        if done[0].item():
            if env.length[0].item() == (Config.GRID_H * Config.GRID_W):
                print("\n[!] 🏆 PERFECT GAME DETECTED 🏆")
                print("    Grid saturation achieved. Agent has won the game.")
            else:
                print("\n[!] AGENT TERMINATED")

            print(f"    Final Score: {score}")
            break

        time.sleep(0.01) # Adjust rendering speed here (lower is faster)

if __name__ == "__main__":
    visualize_best_model()

 HYPERSNAKE-AI | SCORE: 087 | STEP: 0633
                              
                              
                              
                              
                              
                              
                              
                              
                              

[!] 🏆 PERFECT GAME DETECTED 🏆
    Grid saturation achieved. Agent has won the game.
    Final Score: 87
